<h5> Preprocessing The Numerical Dataset (X variables) (9,391 number of features) (Weekly), The data is taken from the lab's dataset

In [ ]:
# Imports necessary packages
import pandas as pd

# Loads the macroeconomic dataset 
macroeconomicPath = "Data (Preprocessing)/Numerical(Source).csv"
macroeconomicData = pd.read_csv(macroeconomicPath) # Shape (537, 75129) # (75129-1) / 8 weeks = 9391 features

# Drop columns that end with (-1), (-2), (-3), (-4), (-5), (-6), (-7)
macroeconomicData = macroeconomicData.loc[:, ~macroeconomicData.columns.str.endswith(('(-1)', '(-2)', '(-3)', '(-4)', '(-5)', '(-6)', '(-7)'))]
# Shape (537, 9392)

# Filters rows to include only data between 2015 and 2019 (a 5 years period)
macroeconomicData = macroeconomicData[
    (macroeconomicData['DATE'] >= '2014-12-29') & (macroeconomicData['DATE'] <= '2020-01-03')
]
# There are 262 weeks between Monday, December 29th, 2014 and Friday, January 3rd, 2020
# All the prices are taken from the Friday's prices
# Shape (262, 9392)

# Renames columns to remove '_(-0)'
macroeconomicData.columns = macroeconomicData.columns.str.replace(r'\s*-\s*\(-0\)', '', regex=True)

# Saves the dataframe into a .csv file
macroeconomicData.to_csv("Data (Preprocessing)/Numerical(Preprocessed).csv", index=False)

print(macroeconomicData.shape) # Shape [262, 9392]

<h5> Prepocessing The Macroeconomics Dataset (y variable) (WTI (Trend) Crude Oil Spot Prices) (Daily)

In [ ]:
# Imports necessary packages
import pandas as pd
from statsmodels.tsa.seasonal import STL

# Loads the daily WTI crude oil price dataset from the a csv file (2016 until 2020) (a 5 years period)
inputPath = 'Crude Oil WTI Spot Price (5 Years) - Copy.csv'
priceData = pd.read_csv(inputPath) # Shape [1254, 2] # The 'Date' and 'Price' columns

# Transforms the 'Date' column into executable format (for filling the missing values)
# Transforms the 'Date' column into the index for the dataframe
priceData['Date'] = pd.to_datetime(priceData['Date'], format='%d-%b-%y')
priceData.set_index('Date', inplace=True) # Shape [1254, 1]

# Fills and handles the missing values (holidays on business days and weekends)
allDates = pd.date_range(start=priceData.index.min(), end=priceData.index.max(), freq='D') # 'D' means all avilable days
priceData = priceData.reindex(allDates)
priceData.index.name = 'Date'
priceData['Price'] = priceData['Price'].interpolate(method='linear').round(2) # Shape [1825, 1]
# The day of the date starts from Monday and ends on Friday (1825 days + 2 days = 1827 / 7 = 261 weeks)

# Removes the last 5 rows because of the incompleteness (no saturday and sunday on that week)
priceData = priceData.iloc[:-5, :] # Shape [1820, 1]

# Calls the price decomposition package (decomposing into trend, seasonal, and residual components)
stl = STL(priceData['Price'], period=28) # 28 represents the number of days in a month 
result = stl.fit()
priceData['trend'] = result.trend
priceData['seasonal'] = result.seasonal
priceData['residual'] = result.resid

# Rounds the trend, seasonal, and residual components to two decimal places after for simplicity
priceData['trend'] = priceData['trend'].round(2)
priceData['seasonal'] = priceData['seasonal'].round(2)
priceData['residual'] = priceData['residual'].round(2)
# Shape [1820, 4]

# Converts the index into 'DATE' column
priceData = priceData.reset_index()
# Renames 'Date' column to 'DATE'
priceData = priceData.rename(columns={'Date': 'DATE'}) # Shape [1820, 5]

# Drops the unnecessary columns
priceData.drop(columns=['Price', 'seasonal', 'residual'], inplace=True) # Shape [1820, 2]

# Renames 'trend' column to 'Trend'
priceData = priceData.rename(columns={'trend': 'Trend'}) # Shape [1820, 5]

# Saves the dataframe into a .csv file
priceData .to_csv("Data (Preprocessing)/TrendDailyFiltered.csv", index=False)

print(priceData.shape) # Shape [1820, 2]

<h5> Prepocessing The Numerical Dataset (y variable) (19 number of crude oil features) (Daily)

In [ ]:
# Imports necessary packages
import pandas as pd

# Gets the universal path for all crude oil datasets
path = "Data (Preprocessing)/"
# Defines file path for each dataset
file_paths = {
    "Crude Oil WTI Futures Historical Data": path + "Crude Oil WTI Futures Historical Data.csv",
    "Brent Oil Futures Historical Data": path + "Brent Oil Futures Historical Data.csv",
    "Dubai Crude Oil (Platts) Financial Futures Historical Data - Price": path + "Dubai Crude Oil (Platts) Financial Futures Historical Data.csv",
    "DCOILBRENTEU": path + "DCOILBRENTEU.csv"
}

# Loads the datasets into dataframes
dataframes = {name: pd.read_csv(path) for name, path in file_paths.items()}

# Defines a function to convert 'K' and 'M' into numerical values
def convertVolume(value):
    if isinstance(value, str):
        value = value.replace(",", "").strip()
        if value.endswith("K"):
            return float(value.replace("K", "")) * 1_000
        elif value.endswith("M"):
            return float(value.replace("M", "")) * 1_000_000
    return pd.to_numeric(value, errors="coerce")

# Converts the Date columns to datetime format and set as index for interpolation
for name, df in dataframes.items():
    # Identifies the correct date column name
    dateCol = "Date" if "Date" in df.columns else "observation_date"
    # Converts date column to datetime format
    df[dateCol] = pd.to_datetime(df[dateCol], errors='coerce')
    # Sets the date column as index
    df.set_index(dateCol, inplace=True)
    # Ensures 'Vol.' is numerical by converting 'K' and 'M'
    if "Vol." in df.columns:
        df["Vol."] = df["Vol."].apply(convertVolume)
    # Ensures 'Change %' is numerical by removing '%' and converting to float
    if "Change %" in df.columns:
        df["Change %"] = df["Change %"].str.replace("%", "", regex=False).astype(float)
    # Creates a full date range from January 5, 2015, to January 3, 2020
    fullDateRange = pd.date_range(start="2015-01-05", end="2020-01-03", freq="D")
    # Reindexs the dataframe to include all dates, filling missing rows with NaN
    df = df.reindex(fullDateRange)
    # Performs interpolation to fill missing values
    df.interpolate(method="linear", inplace=True)
    # Fixs first and last missing values
    df.ffill(inplace=True)  # Fill first missing values with the nearest previous value
    df.bfill(inplace=True)  # Fill last missing values with the nearest next value
    # Renames columns to avoid duplicates
    df = df.add_prefix(f"{name}_")
    # Stores back the processed dataframe
    dataframes[name] = df

# Displays the shape of each dataframe after interpolation
# for name, df in dataframes.items():
    # print(f"Dataset: {name} (after interpolation)")
    # print(df.shape) # Shape [1825, 6] and [1825, 1]

# Combines all dataframes into a single dataframe
merged_df = pd.concat(dataframes.values(), axis=1)

# Ensures the date index is reset and included as a column
merged_df.reset_index(inplace=True)
# Renames the index column to 'DATE'
merged_df.rename(columns={"index": "DATE"}, inplace=True)

# Removes the last 5 rows because of the incompleteness (no saturday and sunday on that week)
merged_df = merged_df.iloc[:-5, :] # Shape [1820, 20]

# Saves the dataframe into a .csv file
merged_df .to_csv("Data (Preprocessing)/Oil(Preprocessed).csv", index=False)

print(merged_df.shape) # Shape [1820, 20]

<h5> Preprocessing The Macroeconomics Dataset (X variables) (9,372 number of features) (Daily)

In [ ]:
# Imports necessary packages
import pandas as pd

# Loads the weekly filtered macroeconomic dataset 
macroeconomicWeeklyPath = "Data (Preprocessing)/Numerical(Preprocessed).csv"
macroeconomicWeeklyData = pd.read_csv(macroeconomicWeeklyPath) # Shape [262, 9392]

# Drops the first 20 columns because of the avilable daily datasetes
macroeconomicWeeklyData = macroeconomicWeeklyData.drop(macroeconomicWeeklyData.columns[1:20], axis=1) # Shape [262, 9373]

# Converts the first column to datetime and set as index
dateColumn = macroeconomicWeeklyData.columns[0]
macroeconomicWeeklyData[dateColumn] = pd.to_datetime(macroeconomicWeeklyData[dateColumn], errors='coerce')
macroeconomicWeeklyData.set_index(dateColumn, inplace=True)
fullDateRange = pd.date_range(start="2014-12-29", end="2020-01-03", freq="D")
# Reindexs to include all dates and interpolate missing values
macroeconomicWeeklyData = macroeconomicWeeklyData.reindex(fullDateRange)
# Performs interpolation to fill missing values
macroeconomicWeeklyData.interpolate(method="linear", inplace=True)
# Fixs first and last missing values
macroeconomicWeeklyData.ffill(inplace=True)  # Fill first missing values with the nearest previous value
macroeconomicWeeklyData.bfill(inplace=True)  # Fill last missing values with the nearest next value

# Ensures the date index is reset and included as a column
macroeconomicWeeklyData.reset_index(inplace=True)
# Renames the index column to 'DATE'
macroeconomicWeeklyData.rename(columns={"index": "DATE"}, inplace=True)  # Shape [1832, 9373]

# Removes the last 5 rows because of the incompleteness (no saturday and sunday on that week)
macroeconomicWeeklyData = macroeconomicWeeklyData.iloc[:-5, :] # Shape [1827, 9373]
# Removes the first 7 rows (starts from 2015, 5th January)
macroeconomicWeeklyData = macroeconomicWeeklyData.iloc[7:, :] # Shape [1820, 9373]

# Saves the dataframe into a .csv file
macroeconomicWeeklyData .to_csv("Data (Preprocessing)/Macroeconomics(Preprocessed).csv", index=False)

print(macroeconomicWeeklyData.shape) # Shape [1820, 9373]

<h5> Preprocessing The Macroeconomics Dataset (X variables) (9,390 number of features) (Daily) (Combine)

In [9]:
# Imports necessary packages
import pandas as pd 

# Loads the two daily macroeconomics datasets
firstPath = "Data (Preprocessing)/Oil(Preprocessed).csv"
firstData = pd.read_csv(firstPath) # Shape [1820, 20]
secondPath = "Data (Preprocessing)/Macroeconomics(Preprocessed).csv"
secondData = pd.read_csv(secondPath) # Shape [1820, 9373]

# Combines the two dataframes based on the 'DATE' column
thirdData = pd.merge(firstData, secondData, on="DATE", how="inner") # Shape [1820, 9392]

# Drop the column "POILWTIUSDM"
thirdData = thirdData.drop(columns=["POILWTIUSDM - POILWTIUSDM"]) # Shape [1820, 9391]

# Saves the dataframe into a .csv file
thirdData .to_csv("Data (Preprocessing)/MacroeconomicsDailyFiltered.csv", index=False)

print(thirdData.shape) # Shape [1820, 9391]

(1820, 9391)
